# Embarrassingly parallel lab: environment & synthetic data

**Run this notebook once** before `workspace_parallel_spcs_demo.ipynb`.

**All runtime names** (database, schemas, warehouse, compute pool, image URI, stage, queue table)
come from **`parallel_lab` in `snowflaker_parallel_spcs_config.yaml`** — edit that file once;
the helper `parallel_lab_config.py` loads it and exposes values to R via environment variables.

**Creates / verifies:**
- Database and schemas from `parallel_lab`
- Optional `SERIES_EVENTS` when `create_synthetic_series_table: true` in YAML
- When `false`: **reuse** existing `SOURCE_DATA` (e.g. `SALES_DATA`)
- `JOB_MANIFEST` and `TRAINING_RESULTS` shells for monitoring

While long jobs run, use `workspace_parallel_spcs_monitor.ipynb` or Streamlit in another tab.
See `PARALLEL_SPCS_DEMO.md`.


## 1. Install R environment & load parallel_lab config


In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(
    config="snowflaker_parallel_spcs_config.yaml",
    packages=["snowflakeR", "RSnowflake"]
)
import parallel_lab_config as plc
import parallel_spcs_workflow as psw
plc.ensure_notebook_path_on_sys_path()
LAB = plc.apply_parallel_lab_environment(plc.load_parallel_lab_dict())
NAMES = psw.ParallelLabNames.from_cfg(LAB)
forbidden = NAMES.contains_forbidden_refs()
if forbidden:
    raise ValueError(f"parallel_lab must not include account-specific refs: {forbidden}")
print("parallel_lab database:", LAB["database"])
print("compute_pool:", LAB.get("compute_pool") or "(unset — set in YAML for doSnowflake)")
print("config file:", LAB.get("_config_path"))


## 2. Verify R packages


In [ ]:
%%R
library(snowflakeR)
library(foreach)
library(iterators)
library(forecast)
cat("snowflakeR", as.character(packageVersion("snowflakeR")), "\n")


## 3. Database layout (from `parallel_lab` in YAML)

| Key | Env var |
|-----|---------|
| Database | `PARALLEL_LAB_DATABASE` |
| Source schema | `PARALLEL_LAB_SCHEMA_SOURCE_DATA` |
| Config schema | `PARALLEL_LAB_SCHEMA_CONFIG` |
| Models schema | `PARALLEL_LAB_SCHEMA_MODELS` |


In [ ]:
%%R
ep_db     <- Sys.getenv("PARALLEL_LAB_DATABASE")
ep_source <- Sys.getenv("PARALLEL_LAB_SCHEMA_SOURCE_DATA")
ep_config <- Sys.getenv("PARALLEL_LAB_SCHEMA_CONFIG")
ep_models <- Sys.getenv("PARALLEL_LAB_SCHEMA_MODELS")
wh        <- Sys.getenv("PARALLEL_LAB_WAREHOUSE")
conn <- sfr_connect()
if (nzchar(wh)) {
  sfr_execute(conn, paste("USE WAREHOUSE", wh))
}
sfr_execute(conn, paste("CREATE DATABASE IF NOT EXISTS", ep_db))
for (sch in c(ep_source, ep_config, ep_models)) {
  sfr_execute(conn, paste("CREATE SCHEMA IF NOT EXISTS", paste(ep_db, sch, sep = ".")))
}
conn <- sfr_use(conn, database = ep_db, schema = ep_source)
rcat(paste("Database", ep_db, "schemas ready."))


## 4. Source series data

If `create_synthetic_series_table` is **true** in YAML, we materialise `SERIES_EVENTS`. If **false**, we skip it and you use existing tables such as **`SALES_DATA`** (`SKU`, `SALE_DATE`, `SALES`).


In [ ]:
%%R
do_syn <- identical(Sys.getenv("PARALLEL_LAB_CREATE_SYNTHETIC_SERIES"), "1")
tbl_events <- paste(ep_db, ep_source, "SERIES_EVENTS", sep = ".")
if (!do_syn) {
  sales_guess <- paste(ep_db, ep_source, "SALES_DATA", sep = ".")
  rcat(paste(
    "Skipping SERIES_EVENTS (parallel_lab.create_synthetic_series_table = false).",
    "Use existing source tables such as", sales_guess
  ))
} else {
  ddl <- paste0(
    "CREATE OR REPLACE TABLE ", tbl_events, " AS ",
    "WITH units AS ( ",
    "  SELECT LPAD(TO_VARCHAR(SEQ4()), 6, '0') AS UNIT_ID ",
    "  FROM TABLE(GENERATOR(ROWCOUNT => 120)) ",
    "), days AS ( ",
    "  SELECT SEQ4() AS d FROM TABLE(GENERATOR(ROWCOUNT => 365)) ",
    ") ",
    "SELECT u.UNIT_ID, ",
    "       DATEADD('day', d.d, DATE '2023-01-01') AS OBS_DATE, ",
    "       10 + MOD(ABS(HASH(u.UNIT_ID, d.d)), 80) * 0.05 + SIN(d.d / 25.0) AS Y ",
    "FROM units u CROSS JOIN days d"
  )
  sfr_execute(conn, ddl)
  rcat(paste("Created", tbl_events))
}


## 5. Monitoring table shells


In [ ]:
%%R
jm <- paste(ep_db, ep_config, "JOB_MANIFEST", sep = ".")
tr <- paste(ep_db, ep_models, "TRAINING_RESULTS", sep = ".")
sfr_execute(conn, paste0(
  "CREATE OR REPLACE TABLE ", jm, " (",
  "JOB_ID VARCHAR, JOB_TYPE VARCHAR, UNIT_ID VARCHAR, STATUS VARCHAR,",
  "WORKER_INDEX INTEGER, STARTED_AT TIMESTAMP_NTZ, COMPLETED_AT TIMESTAMP_NTZ,",
  "ERROR_MSG VARCHAR, UPDATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP())"
))
sfr_execute(conn, paste0(
  "CREATE OR REPLACE TABLE ", tr, " (",
  "JOB_ID VARCHAR, UNIT_ID VARCHAR, WORKER_INDEX INTEGER,",
  "RMSE FLOAT, MAE FLOAT, AIC FLOAT, N_OBS INTEGER, TRAINING_SECS FLOAT,",
  "COMPLETED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP())"
))
rcat("JOB_MANIFEST and TRAINING_RESULTS ready (empty).")


## Done

Open `workspace_parallel_spcs_demo.ipynb`.
